In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ---------------- 설정 ----------------
# 'asof'  : merge_asof backward. 2025 test 행이 가장 최근(2024) 트랙맨 값을 받는다.
#           학습/추론이 동일한 규칙을 쓰므로 원칙적으로 더 타당하다.
# 'exact' : (season, month) 정확 일치 + fillna(0). 900점 버전과 완전히 동일한 동작
#           (트랙맨에 2025가 없어 test에서는 전부 0이 된다).
TRACKMAN_MODE = 'asof'

N_SPLITS = 10                 # 5 -> 10 (각 fold가 90%를 학습, 평균 대상도 늘어 분산 감소)
SEEDS = [42, 202, 2024]       # seed 앙상블 (3개) -> 총 N_SPLITS * len(SEEDS) = 30개 모델
N_OPTUNA_TRIALS = 40          # 하이퍼파라미터 탐색 횟수

# --- 2026-08-18 추가 (2024 시즌 홀드아웃 3-seed 짝지어 검증 결과 반영) ---
# 조건부 투수통계: 기준선 대비 +20~27점 (3 seed 전부 우세, 분산도 ±18->±6로 감소)
USE_COND_STATS = True
# 재중심화: 11개 설정 전부에서 +11~22점 (평균 +18)
RECENTER = True
HOLDOUT_SEASON = 2024         # 오프셋 측정용 홀드아웃 시즌 (이 시즌은 학습에서 빼고 1회 측정)
N_HOLDOUT_FOLDS = 3
# 죽은 피처: asof_pitcher_n이 '경기내'가 아니라 '커리어 누적'이라 의도대로 동작하지 않음
#   is_long_relief 는 전체의 86%(이닝>1 중 97%)로 사실상 inning>1 과 동일,
#   is_strict_inherited_runner 는 0.05%로 상수, pitches_per_inning 은 커리어투구수/이닝.
#   (효과는 +4점 수준으로 미미하나 코드 정합성 차원에서 제거)
DEAD_FEATURES = ['is_long_relief', 'is_short_relief',
                 'is_strict_inherited_runner', 'pitches_per_inning']
print(f"TRACKMAN_MODE = {TRACKMAN_MODE} | N_SPLITS = {N_SPLITS} | SEEDS = {SEEDS}")

# v5: Optuna 재탐색을 끈다. 다시 탐색하면 파라미터가 바뀌어 리더보드 차이가
# '트랙맨 v2 효과'인지 '파라미터 변화'인지 구분되지 않는다 (v4 때 실제로 겪음).
# 아래는 v4(987.3936) 실행에서 나온 값 그대로.
RUN_OPTUNA = False
V4_BEST_PARAMS = {
    "learning_rate": 0.022831883708228414,
    "depth": 8,
    "l2_leaf_reg": 8.552069332567962,
    "bagging_temperature": 0.05636104060100738,
    "random_strength": 0.7731135614050382
}


In [ ]:
STEPS_SRC = r"""
def step1_basic_features(df):
    df_proc = df.copy()
    df_proc['is_weekend_day_game'] = np.where(
        (df_proc['game_month'].isin([4, 5, 9, 10])) & (df_proc['game_dayofweek'].isin([5, 6])), 1.0, 0.0)
    df_proc['is_heat_wave_game'] = np.where(df_proc['game_month'].isin([7, 8]), 1.0, 0.0)
    return df_proc


def step2_pitcher_role_features(df):
    df_proc = df.copy()
    df_proc['is_pure_starter'] = np.where(df_proc['inning'] == 1, 1.0, 0.0)
    df_proc['is_long_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] >= (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    df_proc['is_short_relief'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < (df_proc['inning'] - 1) * 12), 1.0, 0.0)
    return df_proc


def step3_matchup_features(df):
    df_proc = df.copy()
    if 'pitcher_hand' in df_proc.columns and 'batter_hand' in df_proc.columns:
        df_proc['is_same_hand'] = np.where(df_proc['pitcher_hand'] == df_proc['batter_hand'], 1.0, 0.0)
    return df_proc


def step4_refined_count_features(df):
    df_proc = df.copy()
    b, s = df_proc['balls_before'], df_proc['strikes_before']
    df_proc['is_first_pitch'] = np.where((b == 0) & (s == 0), 1.0, 0.0)
    df_proc['is_full_count'] = np.where((b == 3) & (s == 2), 1.0, 0.0)
    pitcher_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    batter_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neutral = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    df_proc['count_advantage'] = np.select(
        [pitcher_ahead, batter_ahead, neutral], ['Pitcher', 'Batter', 'Neutral'], default='None')
    df_proc['is_waste_pitch_sit'] = np.where(((b == 0) & (s == 2)) | ((b == 1) & (s == 2)), 1.0, 0.0)
    df_proc['is_must_strike_sit'] = np.where(((b == 3) & (s == 0)) | ((b == 3) & (s == 1)), 1.0, 0.0)
    return df_proc


def step5_pitches_per_inning(df):
    df_proc = df.copy()
    df_proc['pitches_per_inning'] = df_proc['asof_pitcher_n'] / df_proc['inning'].clip(lower=1)
    return df_proc


def step6_combined_runner_features(df):
    df_proc = df.copy()
    df_proc['is_risp'] = df_proc['base_state'].astype(str).apply(
        lambda x: 1.0 if ('2' in x) or ('3' in x) else 0.0)
    df_proc['is_strict_inherited_runner'] = np.where(
        (df_proc['inning'] > 1) & (df_proc['asof_pitcher_n'] < 5) & (df_proc['num_runners_on'] > 0), 1.0, 0.0)
    df_proc['is_self_risp'] = np.where(
        (df_proc['asof_pitcher_n'] >= 15) & (df_proc['is_risp'] == 1.0), 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['risp_pressure_index'] = df_proc['is_risp'] * li_filled
    df_proc['is_steal_threat_sit'] = np.where(
        (df_proc['runner_on_1b'] == 1) & (df_proc['runner_on_2b'] == 0)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step7_bayesian_smoothing(df, prior_mean=0.64):
    df_proc = df.copy()
    C = 50
    if 'asof_pitcher_success_rate' in df_proc.columns and 'asof_pitcher_n' in df_proc.columns:
        n = df_proc['asof_pitcher_n']
        curr = df_proc['asof_pitcher_success_rate']
        df_proc['smoothed_pitcher_success_rate'] = (n * curr + C * prior_mean) / (n + C)
    return df_proc


def step8_batter_toughness_features(df):
    df_proc = df.copy()
    if 'asof_batter_success_rate' in df_proc.columns and 'asof_batter_middle_rate' in df_proc.columns:
        df_proc['tough_batter_index'] = (1.0 - df_proc['asof_batter_success_rate']) * (1.0 - df_proc['asof_batter_middle_rate'])
    return df_proc


def step9_garbage_time_features(df):
    df_proc = df.copy()
    df_proc['is_garbage_time'] = np.where(df_proc['score_diff_pitcher_team'].abs() >= 7, 1.0, 0.0)
    df_proc['garbage_time_index'] = df_proc['score_diff_pitcher_team'].abs() / (10 - df_proc['inning']).clip(lower=1)
    return df_proc


def step10_recent_form_momentum(df):
    df_proc = df.copy()
    tc = ['asof_pitcher_prev1_game_success_rate',
          'asof_pitcher_prev3_game_success_rate',
          'asof_pitcher_prev5_game_success_rate']
    if all(c in df_proc.columns for c in tc):
        p1, p3, p5 = df_proc[tc[0]], df_proc[tc[1]], df_proc[tc[2]]
        df_proc['momentum_short'] = p1 - p3
        df_proc['momentum_mid'] = p1 - p5
        df_proc['is_heating_up'] = np.where((p1 > p3) & (p3 > p5), 1.0, 0.0)
        df_proc['is_cooling_down'] = np.where((p1 < p3) & (p3 < p5), 1.0, 0.0)
    return df_proc


def step11_veteran_and_pressure_features(df):
    df_proc = df.copy()
    df_proc['is_rookie'] = np.where(df_proc['asof_pitcher_n'] < 684, 1.0, 0.0)
    df_proc['is_veteran'] = np.where(df_proc['asof_pitcher_n'] > 3725, 1.0, 0.0)
    li_filled = df_proc['li'].fillna(0)
    df_proc['rookie_crisis_risk'] = df_proc['is_rookie'] * li_filled
    df_proc['veteran_clutch_ability'] = df_proc['is_veteran'] * li_filled
    return df_proc


def step12_first_pitch_tendency(df):
    df_proc = df.copy()
    if 'asof_pitcher_fastball_rate' in df_proc.columns and 'asof_pitcher_strike_rate' in df_proc.columns:
        if 'is_first_pitch' in df_proc.columns:
            df_proc['first_pitch_fastball_strike_idx'] = (
                df_proc['is_first_pitch'] * df_proc['asof_pitcher_fastball_rate'] * df_proc['asof_pitcher_strike_rate'])
    return df_proc


def step13_sac_fly_threat(df):
    df_proc = df.copy()
    is_3b = df_proc['base_state'].astype(str).apply(lambda x: 1.0 if '3' in x else 0.0)
    df_proc['is_sac_fly_threat'] = np.where(
        (is_3b == 1.0) & (df_proc['outs_before'] < 2)
        & (df_proc['score_diff_pitcher_team'].abs() <= 3), 1.0, 0.0)
    return df_proc


def step14_convert_to_category(df):
    df_proc = df.copy()
    original_cat_cols = ['pitcher_id', 'batter_id', 'pitcher_team_id', 'batter_team_id',
                         'pitcher_hand', 'batter_hand', 'base_state', 'stadium',
                         'pitch_name', 'top_bottom', 'game_type']
    created_cat_cols = ['is_weekend_day_game', 'is_heat_wave_game', 'is_pure_starter',
                        'is_long_relief', 'is_short_relief', 'is_same_hand', 'is_first_pitch',
                        'is_full_count', 'count_advantage', 'is_waste_pitch_sit',
                        'is_must_strike_sit', 'is_risp', 'is_strict_inherited_runner',
                        'is_self_risp', 'is_steal_threat_sit', 'is_sac_fly_threat',
                        'is_garbage_time', 'is_rookie', 'is_veteran',
                        'is_heating_up', 'is_cooling_down']
    all_cat_cols = [c for c in original_cat_cols + created_cat_cols if c in df_proc.columns]
    for c in all_cat_cols:
        df_proc[c] = df_proc[c].astype('category')
    return df_proc
"""

exec(STEPS_SRC)
print("step1~14 정의 완료")


In [ ]:
MAPPING_SRC = r"""
# pitcher_id <-> pitcher_trackman_id 매핑 재구축.
# 주최측이 준 pitcher_id_mapping.csv 는 구종비율 하나로만 매칭돼 약 91%가 틀렸다
# (시즌간 일관성 1.9%, 2024 커버리지 28%). 여기서 다시 만든다.
#   1단계 팀   : (월 x 요일 x 공수) 63차원 투구량 프로파일 -> 헝가리안.
#                검증 = 10개 팀이 6시즌 내내 같은 프랜차이즈로 대응되는가 (10/10).
#                ※ 월 단위 9차원으로는 실패한다 - 팀별 월간 분포가 거의 같아 비용이 평평해진다.
#   2단계 투수 : 팀-시즌 안에서 등판 프로파일 + 이닝 분포 + 구종배합 + 총투구량. 손은 하드제약.
#                검증 = 교정 전 시즌간 일관성 90.9% (매칭에 시즌간 정보를 안 쓰므로 순환 아님).
# 이 문자열이 단일 소스다. tools/rebuild_pitcher_mapping.py 가 노트북에서 이걸 읽어 쓴다.
from scipy.optimize import linear_sum_assignment

_MINOR_PREFIX = ('MIN_', 'KBO_', 'ACE_')   # 2군 / 올스타 / 기타


def _mp_prep(train_df, trackman_df):
    tr = train_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                   'pitcher_id', 'pitcher_hand', 'pitcher_team_id', 'asof_pitcher_pitchmix_n',
                   'asof_pitcher_fastball_rate', 'asof_pitcher_breaking_rate',
                   'asof_pitcher_offspeed_rate']].copy()
    tm = trackman_df[['season', 'game_month', 'game_dayofweek', 'inning', 'top_bottom',
                      'pitcher_trackman_id', 'pitcher_hand', 'pitcher_team',
                      'pitch_type_group']].copy()
    # 손 코딩이 다르다: train 은 1=Left/2=Right 정수, trackman 은 'Left'/'Right' 문자열
    tr['pitcher_hand'] = tr['pitcher_hand'].map({1: 'L', 2: 'R'})
    tm['pitcher_hand'] = tm['pitcher_hand'].map({'Left': 'L', 'Right': 'R'})
    tr['tb'] = tr['top_bottom']
    tm['tb'] = tm['top_bottom'].map({'Top': 'T', 'Bottom': 'B'})
    tm['grp'] = tm['pitch_type_group'].astype(str).str.lower()
    tm['team'] = tm['pitcher_team'].replace({'SK_WYV': 'SSG_LAN'})   # 2021 개명, 같은 프랜차이즈
    tm['is_major'] = ~tm['pitcher_team'].str.startswith(_MINOR_PREFIX, na=False)
    return tr, tm


def _mp_cells(df, key):
    d = df.assign(c=df['game_month'].astype(str) + '_' +
                    df['game_dayofweek'].astype(str) + '_' + df['tb'])
    return d.pivot_table(index=key, columns='c', aggfunc='size', fill_value=0).astype(float)


def _mp_unit(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)


def _mp_match_teams(tr, tm, seasons):
    major = tm[tm['is_major']]
    rows = []
    for s in seasons:
        pa = _mp_cells(tr[tr['season'] == s], 'pitcher_team_id')
        pb = _mp_cells(major[major['season'] == s], 'team')
        pa, pb = pa.div(pa.sum(1), axis=0), pb.div(pb.sum(1), axis=0)
        cols = sorted(set(pa.columns) & set(pb.columns))
        A, B = pa[cols].values, pb[cols].values
        C = ((A[:, None, :] - B[None, :, :]) ** 2).sum(-1)
        r, c = linear_sum_assignment(C)
        rows += [dict(season=s, tid=pa.index[i], code=pb.index[j]) for i, j in zip(r, c)]
    piv = pd.DataFrame(rows).pivot(index='tid', columns='season', values='code')
    stable = int((piv.nunique(axis=1) == 1).sum())
    print(f"  [팀] 6시즌 내내 동일 프랜차이즈: {stable}/{len(piv)}")
    if stable != len(piv):
        raise RuntimeError("팀 매칭이 시즌 간 불일치.\n" + piv.to_string())
    return piv.iloc[:, 0].to_dict()


def _mp_train_mix(sub):
    '''train 의 누적 asof 비율에서 그 시즌만의 구종배합을 복원'''
    g = sub.sort_values('asof_pitcher_pitchmix_n').groupby('pitcher_id')
    n0 = g['asof_pitcher_pitchmix_n'].first()
    n1 = g['asof_pitcher_pitchmix_n'].last()
    out = {c: g[col].last() * n1 - g[col].first() * n0 for c, col in
           [('fastball', 'asof_pitcher_fastball_rate'),
            ('breaking', 'asof_pitcher_breaking_rate'),
            ('offspeed', 'asof_pitcher_offspeed_rate')]}
    M = pd.DataFrame(out)
    return M.div(M.sum(1).replace(0, np.nan), axis=0)


def build_pitcher_map(train_df, trackman_df):
    tr, tm = _mp_prep(train_df, trackman_df)
    seasons = sorted(tr['season'].unique())
    team_of = _mp_match_teams(tr, tm, seasons)
    tr = tr.assign(team=tr['pitcher_team_id'].map(team_of))
    major = tm[tm['is_major']]
    mixsrc = tm[tm['grp'].isin(['fastball', 'breaking', 'offspeed'])]  # 배합은 2군 포함
    MIX = ['fastball', 'breaking', 'offspeed']
    rows = []
    for s in seasons:
        a_all, b_all = tr[tr['season'] == s], major[major['season'] == s]
        mix_a = _mp_train_mix(a_all)
        ms = mixsrc[mixsrc['season'] == s]
        mix_b = pd.crosstab(ms['pitcher_trackman_id'], ms['grp'], normalize='index')
        for team in sorted(set(team_of.values())):
            a, b = a_all[a_all['team'] == team], b_all[b_all['team'] == team]
            if a.empty or b.empty:
                continue
            Pa, Pb = _mp_cells(a, 'pitcher_id'), _mp_cells(b, 'pitcher_trackman_id')
            Ia = a.assign(i=a['inning'].clip(1, 10)).pivot_table(
                index='pitcher_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            Ib = b.assign(i=b['inning'].clip(1, 10)).pivot_table(
                index='pitcher_trackman_id', columns='i', aggfunc='size', fill_value=0
                ).reindex(columns=range(1, 11), fill_value=0).astype(float)
            cols = sorted(set(Pa.columns) & set(Pb.columns))
            ma = mix_a.reindex(Pa.index).reindex(columns=MIX).fillna(0.34).values
            mb = mix_b.reindex(Pb.index).reindex(columns=MIX).fillna(0.34).values
            ta, tb = Pa.values.sum(1), Pb.values.sum(1)
            c_sched = 1 - _mp_unit(Pa[cols].values) @ _mp_unit(Pb[cols].values).T
            c_inn = ((_mp_unit(Ia.values)[:, None, :] -
                      _mp_unit(Ib.values)[None, :, :]) ** 2).sum(-1)
            c_mix = ((ma[:, None, :] - mb[None, :, :]) ** 2).sum(-1)
            c_tot = (np.log1p(ta)[:, None] - np.log1p(tb)[None, :]) ** 2 * 0.05
            ha = a.groupby('pitcher_id')['pitcher_hand'].first().reindex(Pa.index).values
            hb = b.groupby('pitcher_trackman_id')['pitcher_hand'].first().reindex(Pb.index).values
            C = c_sched + c_inn + 2.0 * c_mix + c_tot + 100 * (ha[:, None] != hb[None, :])
            for i, j in zip(*linear_sum_assignment(C)):
                srt = np.sort(C[i])
                rows.append(dict(season=s, pitcher_id=Pa.index[i],
                                 pitcher_trackman_id=Pb.index[j], cost=C[i, j],
                                 margin=srt[1] - srt[0] if len(srt) > 1 else np.inf,
                                 n_tm=tb[j]))
    res = pd.DataFrame(rows)
    # 트레이드 선수는 여러 팀에서 후보가 나오므로 시즌별 1:1 로 정리
    best = res.sort_values('cost').groupby(['season', 'pitcher_id'], as_index=False).first()
    best = best.sort_values('cost').groupby(['season', 'pitcher_trackman_id'],
                                            as_index=False).first()
    vote = best.groupby(['pitcher_trackman_id', 'pitcher_id'])['n_tm'].sum().reset_index()
    win = (vote.sort_values('n_tm', ascending=False)
              .groupby('pitcher_trackman_id', as_index=False).first()
              .rename(columns={'pitcher_id': 'vote_pid'})[['pitcher_trackman_id', 'vote_pid']])
    best = best.merge(win, on='pitcher_trackman_id')
    # 검증은 반드시 다수결 '이전' 값으로. 교정 후에는 정의상 100%라 증거가 못 된다.
    g = best.groupby('pitcher_trackman_id')['pitcher_id']
    multi = g.nunique()[g.size() > 1]
    print(f"  [검증] 교정 전 시즌간 일관성 {(multi == 1).mean() * 100:.1f}% "
          f"(2시즌+ 등장 {len(multi)}명)")
    print(f"  [투수] 시즌간 다수결 교정 {int((best['pitcher_id'] != best['vote_pid']).sum())} "
          f"/ {len(best)}쌍")
    best['pitcher_id'] = best['vote_pid']
    out = best[['season', 'pitcher_id', 'pitcher_trackman_id', 'cost', 'margin']].copy()
    out['conf'] = np.where(out['cost'] <= out['cost'].quantile(0.75), 'high',
                    np.where(out['cost'] <= out['cost'].quantile(0.90), 'mid', 'low'))
    return out.sort_values(['season', 'pitcher_id']).reset_index(drop=True)
"""

exec(MAPPING_SRC)
print("build_pitcher_map 정의 완료")


In [ ]:
def step15_prep_trackman_data(trackman_df, pitcher_map_df):
    # 매핑에 season 이 있으면 반드시 시즌까지 키로 쓴다. pitcher_trackman_id 단독으로 붙이면
    # 한 투구가 여러 투수에게 중복 귀속돼 1.6배로 팽창한다 (2026-08-19 발견).
    keys = ['season', 'pitcher_trackman_id'] if 'season' in pitcher_map_df.columns \
        else ['pitcher_trackman_id']
    tm = pd.merge(trackman_df, pitcher_map_df[keys + ['pitcher_id']].drop_duplicates(),
                  on=keys, how='inner')
    if len(tm) > len(trackman_df):
        raise RuntimeError(f"트랙맨 병합이 팽창했습니다 ({len(trackman_df):,} -> {len(tm):,}). "
                           "매핑 키를 확인하세요.")
    b, s = tm['balls_before'], tm['strikes_before']
    p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
    b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
    neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
    tm['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                       ['Pitcher', 'Batter', 'Neutral'], default='None')
    tm['pitch_group'] = tm['pitch_type_group'].astype(str).str.lower()
    return tm[tm['pitch_group'].isin(['fastball', 'breaking', 'offspeed'])].copy()


def step16_calc_expected_difficulty(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    sit = tm.groupby(['season', 'game_month', 'pitcher_id', 'count_advantage', 'pitch_group']
                     ).size().unstack(fill_value=0).reset_index()
    for c in groups:
        if c not in sit.columns:
            sit[c] = 0
    sit = sit.sort_values(by=['pitcher_id', 'count_advantage', 'season', 'game_month'])
    g = sit.groupby(['pitcher_id', 'count_advantage'])
    sit['past_fb'] = g['fastball'].cumsum() - sit['fastball']
    sit['past_br'] = g['breaking'].cumsum() - sit['breaking']
    sit['past_off'] = g['offspeed'].cumsum() - sit['offspeed']
    tot = sit['past_fb'] + sit['past_br'] + sit['past_off']
    sit['past_total'] = tot
    sit['exp_fb_prob'] = np.where(tot > 0, sit['past_fb'] / tot, 0)
    sit['exp_br_prob'] = np.where(tot > 0, sit['past_br'] / tot, 0)
    sit['exp_off_prob'] = np.where(tot > 0, sit['past_off'] / tot, 0)

    dm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group'])[['rel_height', 'rel_side']].std()
    dm['diff_score'] = dm['rel_height'] + dm['rel_side']
    dm = dm.reset_index()
    dp = dm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values='diff_score', fill_value=np.nan).reset_index()
    for c in groups:
        if c not in dp.columns:
            dp[c] = 0
    dp = dp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    gd = dp.groupby(['pitcher_id'])
    dp['past_fb_diff'] = gd['fastball'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_br_diff'] = gd['breaking'].transform(lambda x: x.shift(1).expanding().mean())
    dp['past_off_diff'] = gd['offspeed'].transform(lambda x: x.shift(1).expanding().mean())

    res = pd.merge(sit, dp, on=['season', 'game_month', 'pitcher_id'], how='left')
    res['expected_control_difficulty'] = (res['exp_fb_prob'] * res['past_fb_diff']
                                          + res['exp_br_prob'] * res['past_br_diff']
                                          + res['exp_off_prob'] * res['past_off_diff'])
    return res[['season', 'game_month', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']]


def step17_calc_pitch_speed(tm):
    fb = tm[tm['pitch_group'] == 'fastball']
    sp = fb.groupby(['season', 'game_month', 'pitcher_id'])['rel_speed'].mean().reset_index()
    sp = sp.sort_values(by=['pitcher_id', 'season', 'game_month'])
    sp['past_fb_speed_mean'] = sp.groupby(['pitcher_id'])['rel_speed'].transform(
        lambda x: x.shift(1).expanding().mean())
    return sp[['season', 'game_month', 'pitcher_id', 'past_fb_speed_mean']]


def step18_calc_pitch_consistency_by_group(tm):
    groups = ['fastball', 'breaking', 'offspeed']
    metrics = ['rel_height_std', 'rel_side_std', 'extension_std',
               'spin_rate_std', 'vert_break_std', 'horz_break_std']
    cm = tm.groupby(['season', 'game_month', 'pitcher_id', 'pitch_group']).agg(
        rel_height_std=('rel_height', 'std'), rel_side_std=('rel_side', 'std'),
        extension_std=('extension', 'std'), spin_rate_std=('spin_rate', 'std'),
        vert_break_std=('induced_vert_break', 'std'), horz_break_std=('horz_break', 'std')
    ).reset_index()
    pv = cm.pivot_table(index=['season', 'game_month', 'pitcher_id'],
                        columns='pitch_group', values=metrics, fill_value=np.nan)
    pv.columns = [f"{grp}_{val}" for val, grp in pv.columns]
    pv = pv.reset_index()
    for pg in groups:
        for m in metrics:
            if f"{pg}_{m}" not in pv.columns:
                pv[f"{pg}_{m}"] = np.nan
    pv = pv.sort_values(by=['pitcher_id', 'season', 'game_month'])
    g = pv.groupby(['pitcher_id'])
    out_cols = ['season', 'game_month', 'pitcher_id']
    for pg in groups:
        for m in metrics:
            src, dst = f"{pg}_{m}", f"past_{pg}_{m}"
            pv[dst] = g[src].transform(lambda x: x.shift(1).expanding().mean())
            out_cols.append(dst)
    return pv[out_cols]


# ================= 조건부 투수통계 (2026-08-18 추가) =================
# 설계: 성공률이 매 시즌 단조 하락(.565->.486)하므로 원시 성공률을 그대로 쓰면 과거 시즌의
#       높은 수준이 그대로 섞여 들어온다. 그래서 '그 시즌 리그평균 대비 편차'로 디트렌드한 뒤
#       0(=리그평균)으로 shrink 하는 경험적 베이즈 방식을 쓴다.
#       표본이 적은 조합일수록 자동으로 0에 가까워지므로 콜드스타트도 자연히 처리된다.
# 검증: 2024 홀드아웃 3-seed 짝지어 비교에서 기준선 대비 +20(원본)/+27(재중심화)
COND_SPECS = [
    (['pitcher_id'],                                    200, 'cond_p'),
    (['pitcher_id', 'count_advantage'],                 100, 'cond_pc'),
    (['pitcher_id', 'batter_hand'],                     100, 'cond_ph'),
    (['pitcher_id', 'batter_hand', 'count_advantage'],   50, 'cond_phc'),
]


def _add_dev(df):
    """control_success 를 '그 시즌 리그평균 대비 편차'로 변환 (드리프트 제거)."""
    lg = df.groupby('season')['control_success'].mean()
    return df['control_success'] - df['season'].map(lg)


def build_cond_table(src, keys, C, name):
    g = src.groupby(keys, observed=True)['_dev'].agg(['sum', 'count']).reset_index()
    g[name] = g['sum'] / (g['count'] + C)          # 0(리그평균)으로 shrink
    return g[keys + [name]]


def attach_cond_features(df):
    """학습용: 각 행은 '그 시즌보다 과거' 데이터로만 인코딩 -> leak-free.
    (배포 시 2025 test 가 2019~2024 로 인코딩되는 것과 동일한 규칙)"""
    df = df.copy()
    df['_dev'] = _add_dev(df)
    seasons = sorted(df['season'].unique())
    for keys, C, name in COND_SPECS:
        col = np.full(len(df), np.nan)
        for s in seasons:
            past = df[df['season'] < s]
            if len(past) == 0:
                continue
            t = build_cond_table(past, keys, C, name).set_index(keys)[name]
            cur = (df['season'] == s).values
            sl = df.loc[cur, keys]
            idx = pd.MultiIndex.from_frame(sl) if len(keys) > 1 else pd.Index(sl[keys[0]])
            col[cur] = t.reindex(idx).values
        df[name] = col
        print(f"  {name}: 결측 {np.isnan(col).mean()*100:.1f}% (첫 시즌 + 신규투수)")
    return df.drop(columns=['_dev'])


def build_all_cond_tables(df):
    """추론용: 학습 전 시즌을 다 써서 만든 최종 룩업 테이블."""
    d = df.copy()
    d['_dev'] = _add_dev(d)
    return {name: build_cond_table(d, keys, C, name) for keys, C, name in COND_SPECS}


COND_COLS = [name for _, _, name in COND_SPECS]


In [ ]:
def run_full_pipeline(train_df, trackman_df, pitcher_map, trackman_mode='asof'):
    print(f"파이프라인 시작 (trackman_mode={trackman_mode})...")
    df_proc = train_df.copy()

    df_proc = step1_basic_features(df_proc)
    df_proc = step2_pitcher_role_features(df_proc)
    df_proc = step3_matchup_features(df_proc)
    df_proc = step4_refined_count_features(df_proc)
    df_proc = step5_pitches_per_inning(df_proc)
    df_proc = step6_combined_runner_features(df_proc)

    prior_mean = float(df_proc['asof_pitcher_success_rate'].mean())
    print(f"  prior_mean = {prior_mean:.6f}")

    df_proc = step7_bayesian_smoothing(df_proc, prior_mean=prior_mean)
    df_proc = step8_batter_toughness_features(df_proc)
    df_proc = step9_garbage_time_features(df_proc)
    df_proc = step10_recent_form_momentum(df_proc)
    df_proc = step11_veteran_and_pressure_features(df_proc)
    df_proc = step12_first_pitch_tendency(df_proc)
    df_proc = step13_sac_fly_threat(df_proc)

    if 'count_advantage' not in df_proc.columns:
        b, s = df_proc['balls_before'], df_proc['strikes_before']
        p_ahead = ((b == 0) & (s == 1)) | ((b == 0) & (s == 2)) | ((b == 1) & (s == 2))
        b_ahead = ((b == 1) & (s == 0)) | ((b == 2) & (s == 0)) | ((b == 3) & (s == 0)) | ((b == 2) & (s == 1)) | ((b == 3) & (s == 1))
        neu = ((b == 1) & (s == 1)) | ((b == 2) & (s == 2))
        df_proc['count_advantage'] = np.select([p_ahead, b_ahead, neu],
                                                ['Pitcher', 'Batter', 'Neutral'], default='None')

    tm_base = step15_prep_trackman_data(trackman_df, pitcher_map)
    feat_diff = step16_calc_expected_difficulty(tm_base)
    feat_speed = step17_calc_pitch_speed(tm_base)
    feat_rp = step18_calc_pitch_consistency_by_group(tm_base)
    rp_value_cols = [c for c in feat_rp.columns if c.startswith('past_')]

    if trackman_mode == 'asof':
        for f in [feat_diff, feat_speed, feat_rp]:
            f['time_idx'] = f['season'] * 100 + f['game_month']
            f.sort_values('time_idx', inplace=True)
        df_proc['time_idx'] = df_proc['season'] * 100 + df_proc['game_month']
        df_proc = df_proc.sort_values('time_idx')

        df_proc = pd.merge_asof(
            df_proc,
            feat_diff[['time_idx', 'pitcher_id', 'count_advantage', 'expected_control_difficulty']],
            on='time_idx', by=['pitcher_id', 'count_advantage'], direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_speed[['time_idx', 'pitcher_id', 'past_fb_speed_mean']],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = pd.merge_asof(
            df_proc, feat_rp[['time_idx', 'pitcher_id'] + rp_value_cols],
            on='time_idx', by='pitcher_id', direction='backward')
        df_proc = df_proc.drop(columns=['time_idx'])
    else:
        df_proc = pd.merge(df_proc, feat_diff,
                           on=['season', 'game_month', 'pitcher_id', 'count_advantage'], how='left')
        df_proc = pd.merge(df_proc, feat_speed,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        df_proc = pd.merge(df_proc, feat_rp,
                           on=['season', 'game_month', 'pitcher_id'], how='left')
        for c in ['expected_control_difficulty', 'past_fb_speed_mean'] + rp_value_cols:
            if c in df_proc.columns:
                df_proc[c] = df_proc[c].fillna(0)

    # 조건부 투수통계 (step14 이전에 붙여야 함: pitcher_id/count_advantage 가 아직 원시 dtype)
    cond_tables = {}
    if USE_COND_STATS:
        print("조건부 투수통계 생성...")
        df_proc = attach_cond_features(df_proc)
        cond_tables = build_all_cond_tables(df_proc)   # 추론용 최종 테이블(전 시즌)

    df_proc = step14_convert_to_category(df_proc)
    print("파이프라인 완료.")
    return df_proc.reset_index(drop=True), prior_mean, feat_diff, feat_speed, feat_rp, cond_tables


In [ ]:
DATA_DIR = "/kaggle/input/datasets/homekeggle/aimers/open/data"
df_train = pd.read_csv(f"{DATA_DIR}/train.csv")
df_trackman = pd.read_csv(f"{DATA_DIR}/trackman_history.csv")
pitcher_id_mapping = build_pitcher_map(df_train, df_trackman)
d = run_full_pipeline(df_train, df_trackman, pitcher_id_mapping,
                      trackman_mode=TRACKMAN_MODE)[0]
print("기준 프레임", d.shape)


In [ ]:
# asof_* 재설계 스크리닝 (캐글 GPU)
import gc
import numpy as np, pandas as pd
from catboost import CatBoostRegressor
TRS = [2019, 2020, 2021, 2022, 2023]; VA = 2024

d = d.sort_values('row_id').reset_index(drop=True)
y_all = d['control_success'].to_numpy(float)
pid = d['pitcher_id'].to_numpy()
season = d['season'].to_numpy()
lg = pd.Series(y_all).groupby(season).mean()          # 시즌 리그평균
dev = y_all - pd.Series(season).map(lg).to_numpy()    # 디트렌드된 성공/실패

MINP = 30


def rolling_series(vals, N=None, hl=None):
    """투수별 leak-free 이동평균. N구 단순평균 또는 반감기 hl 지수가중."""
    s = pd.Series(vals).groupby(pid)
    if N is not None:
        return s.transform(lambda x: x.shift(1).rolling(N, min_periods=MINP).mean()).to_numpy()
    return s.transform(lambda x: x.shift(1).ewm(halflife=hl, min_periods=MINP).mean()).to_numpy()


def freeze_by_season(col):
    """각 (투수, 시즌) 행에 '그 시즌 시작 직전' 값을 채운다 -> 추론 시점과 동일한 구조."""
    t = pd.DataFrame({'p': pid, 's': season, 'v': col})
    last = t.groupby(['p', 's']).v.last().reset_index()       # 시즌 말 값
    last['s'] = last['s'] + 1                                  # 다음 시즌이 쓸 값
    m = t[['p', 's']].merge(last, on=['p', 's'], how='left')
    return m.v.to_numpy()


print('피처 생성...', flush=True)
FEATS = {}
for N in (100, 300, 1000):
    FEATS[f'roll{N}'] = rolling_series(y_all, N=N)
FEATS['ewm500'] = rolling_series(y_all, hl=500)
FEATS['roll300_dev'] = rolling_series(dev, N=300)
for k in list(FEATS):
    FEATS[k] = FEATS[k].astype('float32')
FROZEN = {k: freeze_by_season(v).astype('float32') for k, v in FEATS.items()}
va_m = (season == VA)
for k, v in FEATS.items():
    print(f'  {k:<12} 결측 {np.isnan(v).mean()*100:5.1f}%  | 고정판 결측 '
          f'{np.isnan(FROZEN[k]).mean()*100:5.1f}%  2024만 {np.isnan(FROZEN[k][va_m]).mean()*100:5.1f}%',
          flush=True)

# 운영측 asof 가 2024를 얼마나 과대평가하는지 vs 새 피처
act = y_all[va_m].mean()
print(f'\n2024 실제 {act:.4f} | 운영측 asof {np.nanmean(d.asof_pitcher_success_rate.to_numpy()[va_m]):.4f}', flush=True)
for k in FEATS:
    print(f'  {k:<12} fresh {np.nanmean(FEATS[k][va_m]):.4f}  frozen {np.nanmean(FROZEN[k][va_m]):.4f}', flush=True)

# ---------------- 평가 ----------------
base_drop = ['control_success', 'row_id', 'pitcher_id', 'batter_id', '_dev', 'time_idx']
BASE = [c for c in d.columns if c not in base_drop]
X_all = d[BASE].copy()
for c in BASE:
    if X_all[c].dtype.name in ('category', 'object'):
        X_all[c] = X_all[c].astype(str)
cf = [c for c in BASE if X_all[c].dtype == object]

tr_m = np.isin(season, TRS)
yva = y_all[va_m]; r = yva.mean(); naive = r * (1 - r)
tgt = np.polyval(np.polyfit(TRS, lg.loc[TRS].values, 1), VA)
ytr = y_all[tr_m]
sk = lambda q: (1 - ((np.clip(q, 1e-6, 1-1e-6) - yva) ** 2).mean() / naive) * 100000


def recenter(p, t):
    q = np.clip(p, 1e-6, 1-1e-6); lo = np.log(q/(1-q)); o = 0.0
    for _ in range(300):
        c = 1/(1+np.exp(-(lo+o))); e = c.mean()-t
        if abs(e) < 1e-9: break
        o -= e*4
    return 1/(1+np.exp(-(lo+o)))


def fit(extra, seed):
    """extra: {컬럼명: 배열} — 기준선에 추가할 피처"""
    X = X_all
    if extra:
        X = X_all.copy()
        for k, v in extra.items():
            X[k] = v
    m = CatBoostRegressor(loss_function='RMSE', iterations=300, learning_rate=0.1, depth=6,
                          verbose=0, task_type='GPU', random_seed=seed, cat_features=cf)
    m.fit(X[tr_m], ytr)
    p = np.clip(m.predict(X[va_m]), 1e-6, 1-1e-6)
    del X; gc.collect()
    return sk(p), sk(recenter(p, tgt))


def mix(name, frozen):
    """A(fresh) = 학습행 fresh + 검증행 frozen / B(frozen) = 전 행 frozen"""
    v = FEATS[name].copy()
    v[va_m] = FROZEN[name][va_m]
    return FROZEN[name] if frozen else v


CFG = [('0. 기준선',            {}),
       ('A. roll300 fresh',    {'r300': mix('roll300', False)}),
       ('B. roll100 frozen',   {'r100': mix('roll100', True)}),
       ('B. roll300 frozen',   {'r300': mix('roll300', True)}),
       ('B. roll1000 frozen',  {'r1000': mix('roll1000', True)}),
       ('B. ewm500 frozen',    {'e500': mix('ewm500', True)}),
       ('B. roll300 디트렌드',  {'r300d': mix('roll300_dev', True)}),
       ('B. roll100+1000',     {'r100': mix('roll100', True), 'r1000': mix('roll1000', True)})]

print(f'\n기준 피처 {len(BASE)}개 | 설정 {len(CFG)}개 x 3 seed = {len(CFG)*3} fit\n', flush=True)
res = {n: {'raw': [], 'rc': []} for n, _ in CFG}
for s in (42, 7, 2024):
    for n, ex in CFG:
        a, b = fit(ex, s)
        res[n]['raw'].append(a); res[n]['rc'].append(b)
        print(f'  seed {s:>4} | {n:<22} 원본 {a:>6.0f}  재중심화 {b:>6.0f}', flush=True)

    print(f'\n--- seed {s} 까지 누적 (기준 = 0. 기준선) ---', flush=True)
    bR = np.array(res[CFG[0][0]]['raw']); bC = np.array(res[CFG[0][0]]['rc'])
    for n, _ in CFG:
        raw, rc = np.array(res[n]['raw']), np.array(res[n]['rc'])
        line = f'  {n:<22} 원본 {raw.mean():>6.0f}(±{raw.std():>3.0f})  재중심화 {rc.mean():>6.0f}(±{rc.std():>3.0f})'
        if n != CFG[0][0]:
            dr, dc = raw - bR, rc - bC
            line += f'  | 차이 원본 {dr.mean():+.0f}(±{dr.std():.0f}) 재중심화 {dc.mean():+.0f}(±{dc.std():.0f})'
        print(line, flush=True)
    print(flush=True)
print('완료')
